# Vegetable Image Classification using CNN — Complete Assignment

## 1. Imports and reproducibility
This executable notebook covers the required full pipeline: dataset discovery, class distribution, ImageDataGenerator, augmentation, a CNN built **from scratch**, training with EarlyStopping, evaluation, and learning curves. The first code cell imports all libraries. Attach the [Vegetable Image Dataset](https://www.kaggle.com/datasets/misrakahmed/vegetable-image-dataset) as Kaggle input, enable a GPU for model training, and run all cells in order. No pretrained models or invented results are used.

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Discover and validate Kaggle paths
Automatically locate the dataset's train, test and validation folders under Kaggle Input, even if the mounted dataset has an extra nesting level. The code reports the detected root and fails clearly when none or multiple are found.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
SPLITS = ("train", "test", "validation")

def discover_dataset_root(input_root=INPUT_ROOT, max_depth=4):
    """Find a mounted folder containing train, test and validation directories."""
    if not input_root.is_dir():
        raise FileNotFoundError(f"Kaggle input directory is unavailable: {input_root}")

    preferred = input_root / "vegetable-image-dataset" / "Vegetable Images"
    if all((preferred / split).is_dir() for split in SPLITS):
        return preferred

    candidates = []
    pending = [(input_root, 0)]
    while pending:
        folder, depth = pending.pop()
        child_dirs = sorted((p for p in folder.iterdir() if p.is_dir()), key=str)
        if all((folder / split).is_dir() for split in SPLITS):
            candidates.append(folder)
            continue  # Do not inspect class folders or image files.
        if depth < max_depth:
            pending.extend((child, depth + 1) for child in child_dirs)

    if not candidates:
        mounted = sorted(p.name for p in input_root.iterdir())
        raise FileNotFoundError(
            "Could not find a folder containing train, test and validation under "
            f"{input_root}. Mounted inputs: {mounted}. "
            "Expand the Kaggle Input tree and confirm the dataset is attached and extracted."
        )
    if len(candidates) > 1:
        raise RuntimeError(
            "Multiple dataset roots were found. Choose one explicitly: "
            + ", ".join(str(path) for path in sorted(candidates))
        )
    return candidates[0]

DATASET_ROOT = discover_dataset_root()
TRAIN_DIR = DATASET_ROOT / "train"
TEST_DIR = DATASET_ROOT / "test"
VALIDATION_DIR = DATASET_ROOT / "validation"
SPLIT_PATHS = {"Train": TRAIN_DIR, "Test": TEST_DIR, "Validation": VALIDATION_DIR}

for label, path in SPLIT_PATHS.items():
    if not path.is_dir():
        raise NotADirectoryError(f"{label} directory is missing: {path}")

print(f"Detected dataset root: {DATASET_ROOT}")
for label, path in SPLIT_PATHS.items():
    print(f"{label}: {path}")


## 3. Reusable image-counting function
Count supported image filenames directly inside each class directory, case-insensitively. Preserve empty classes; ignore non-image files, split-level files, and nested directories. This measures file counts without decoding or verifying images.

In [ ]:
def count_images(split_dir):
    """Count supported image files immediately inside each class directory."""
    split_dir = Path(split_dir)
    if not split_dir.exists():
        raise FileNotFoundError(f'Split folder does not exist: {split_dir}')
    if not split_dir.is_dir():
        raise NotADirectoryError(f'Split path is not a directory: {split_dir}')
    extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp', '.tif', '.tiff'}
    class_dirs = sorted(path for path in split_dir.iterdir() if path.is_dir())
    if not class_dirs:
        raise ValueError(f'No class folders found in: {split_dir}')
    return {
        class_dir.name: sum(
            1 for path in class_dir.iterdir()
            if path.is_file() and path.suffix.lower() in extensions
        )
        for class_dir in class_dirs
    }

## 4. Count splits independently and build three DataFrames
Independent counts preserve differences in class membership across train, test, and validation.

In [ ]:
train_counts = count_images(TRAIN_DIR)
test_counts = count_images(TEST_DIR)
validation_counts = count_images(VALIDATION_DIR)
train_df = pd.DataFrame(list(train_counts.items()), columns=['Vegetable', 'Train Count'])
test_df = pd.DataFrame(list(test_counts.items()), columns=['Vegetable', 'Test Count'])
validation_df = pd.DataFrame(list(validation_counts.items()), columns=['Vegetable', 'Validation Count'])
for label, frame in [('Train', train_df), ('Test', test_df), ('Validation', validation_df)]:
    print(f'{label} image counts')
    display(frame)

## 5. Merge counts and report totals
Outer joins retain the union of classes. Missing class folders receive zero counts and an explicit warning. The detected class count is the union across all splits.

In [ ]:
count_columns = ['Train Count', 'Test Count', 'Validation Count']
counts_df = (
    train_df.merge(test_df, on='Vegetable', how='outer', validate='one_to_one')
    .merge(validation_df, on='Vegetable', how='outer', validate='one_to_one')
    .sort_values('Vegetable').reset_index(drop=True)
)
counts_df[count_columns] = counts_df[count_columns].fillna(0).astype('int64')
counts_df = counts_df[['Vegetable', *count_columns]]
display(counts_df)
all_classes = set(train_counts) | set(test_counts) | set(validation_counts)
print(f'Detected class count (union): {len(all_classes)}')
for label, observed in [('Train', train_counts), ('Test', test_counts), ('Validation', validation_counts)]:
    print(f'{label}: {len(observed)} class folders; {sum(observed.values())} images')
    missing = sorted(all_classes - set(observed))
    if missing:
        print(f'WARNING: {label} is missing class folders: {missing}')
print(f'Total images across all splits: {int(counts_df[count_columns].to_numpy().sum())}')

## 6. Plot class distribution
Grouped bars compare splits for each vegetable. The vertical axis measures image files and starts at zero; rotated labels improve readability.

In [ ]:
ax = counts_df.set_index('Vegetable')[count_columns].plot(kind='bar', figsize=(14, 6), width=0.8)
ax.set_title('Vegetable image counts by class and dataset split')
ax.set_xlabel('Vegetable')
ax.set_ylabel('Number of images')
ax.set_ylim(bottom=0)
ax.legend(title='Dataset split', labels=['Train', 'Test', 'Validation'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Explain balance using real counts
Balance is assessed across classes within each split; the splits need not have equal sizes. The report uses observed totals, minima, maxima, and ratios. Equal counts mean exactly balanced; unequal counts are reported without an arbitrary imbalance threshold. Zero-count classes prevent a finite maximum/minimum ratio.

In [ ]:
for label, column in zip(['Train', 'Test', 'Validation'], count_columns):
    values = counts_df[column]
    minimum, maximum = int(values.min()), int(values.max())
    total = int(values.sum())
    print(f'\n{label}: {total} images across {len(values)} union classes; minimum = {minimum}, maximum = {maximum}.')
    if total == 0:
        print('No supported images were found; image balance cannot be assessed.')
    elif minimum == maximum:
        print(f'Exactly balanced: every class has {minimum} images.')
    else:
        print('Class counts differ in this split.')
        print('Minimum-count classes:', ', '.join(counts_df.loc[values == minimum, 'Vegetable']))
        print('Maximum-count classes:', ', '.join(counts_df.loc[values == maximum, 'Vegetable']))
        if minimum > 0:
            print(f'Maximum/minimum count ratio: {maximum / minimum:.3f}')
        else:
            print('At least one class has zero images; maximum/minimum ratio is undefined.')

## 8. EDA checkpoint and actual Kaggle observations
The previous real Kaggle EDA run found 15 classes, with 1,000 images per class in train (15,000 total), 200 per class in validation (3,000), and 200 per class in test (3,000). Thus all three splits were balanced by **file count**. These are observations from that earlier run, not fabricated model-training metrics. The live computations in sections 3–7 remain the source of truth if the data changes. Continue to section 9 for preprocessing, CNN training and evaluation.

## 9. Image preprocessing and data augmentation

The assignment explicitly requires `ImageDataGenerator`. Training images are rescaled and augmented (rotation, shifts, shear, zoom, horizontal flip). Validation images are **only rescaled**, so validation measures performance on images that are not randomly transformed. Both iterators use a consistent `validation_split=0.2` of the original **train/** directory. The untouched original **validation/** and **test/** folders remain separate external holdouts.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 15
VALIDATION_SPLIT = 0.20

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=VALIDATION_SPLIT,
)

val_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=VALIDATION_SPLIT,
)

train_ds = train_datagen.flow_from_directory(
    directory=str(TRAIN_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED,
)

val_ds = val_datagen.flow_from_directory(
    directory=str(TRAIN_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED,
)

print(f"Training iterator: {train_ds.samples:,} images")
print(f"Internal validation iterator: {val_ds.samples:,} images")
print("Class indices:", train_ds.class_indices)

## 10. Verify generators and check for split leakage

The class-index mapping must match across iterators. File paths selected for training and internal validation must not overlap. The separate holdout directories are not read by either training iterator. We also inspect the shapes of one batch and show representative augmented images.

In [ ]:
assert train_ds.class_indices == val_ds.class_indices, "Train/validation class maps differ."
assert len(train_ds.class_indices) == NUM_CLASSES, "Expected exactly 15 vegetable classes."
assert set(train_ds.filepaths).isdisjoint(set(val_ds.filepaths)), "Internal split leakage detected."
assert train_ds.samples + val_ds.samples == sum(train_counts.values()), "Unexpected split sizes."

print("Split checks passed.")
print("Training batches:", len(train_ds), "Validation batches:", len(val_ds))
batch_images, batch_labels = next(train_ds)
print("Image batch shape:", batch_images.shape)
print("Label batch shape:", batch_labels.shape)
assert batch_images.shape[1:] == (224, 224, 3)
assert batch_labels.shape[1] == NUM_CLASSES
assert 0 <= batch_images.min() and batch_images.max() <= 1.0 + 1e-6

idx_to_class = {idx: name for name, idx in train_ds.class_indices.items()}
fig, axes = plt.subplots(2, 4, figsize=(12, 7))
for i, ax in enumerate(axes.flat):
    if i >= len(batch_images):
        ax.axis("off")
        continue
    ax.imshow(np.clip(batch_images[i], 0, 1))
    ax.set_title(idx_to_class[int(np.argmax(batch_labels[i]))])
    ax.axis("off")
plt.suptitle("Augmented training images (rescaled to [0, 1])")
plt.tight_layout()
plt.show()
train_ds.reset()

## 11. Build the required CNN from scratch (Sequential)

Three convolutional blocks learn local visual patterns at increasingly abstract scales: **32**, **64**, and **128** filters, all with 3×3 kernels and ReLU activations. MaxPooling reduces spatial resolution and computation. Flatten converts the learned feature maps to a vector; a Dense(512) layer learns combinations of features. Dropout(0.5) regularizes training by randomly deactivating units. The 15-way Softmax output represents class probabilities that sum to one. This architecture follows the assignment, rather than transfer learning.

In [ ]:
model = Sequential([
    Input(shape=(224, 224, 3)),
    Conv2D(32, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(512, activation="relu"),
    Dropout(0.5),
    Dense(NUM_CLASSES, activation="softmax"),
])

model.summary()
assert sum(isinstance(layer, Conv2D) for layer in model.layers) == 3
assert model.output_shape[-1] == NUM_CLASSES

## 12. Compile the model

Adam adjusts each weight's learning rate during optimization. Categorical crossentropy is appropriate for 15-class, one-hot labels produced by `class_mode="categorical"`. Accuracy measures the fraction of images classified correctly.

In [ ]:
model.compile(
    optimizer=Adam(),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
print("Model compiled: Adam / categorical_crossentropy / accuracy")

## 13. Train the CNN — 20 epochs maximum with EarlyStopping

We monitor `val_loss` to detect worsening performance on unseen internal-validation images. With `patience=5`, training stops if validation loss fails to improve for five consecutive epochs; `restore_best_weights=True` restores the weights from the best validation-loss epoch. This saves unnecessary computation and can limit overfitting. **GPU acceleration is recommended:** the assignment's required Flatten → Dense(512) model has many parameters, so CPU training may be slow.

In [ ]:
MAX_EPOCHS = 20
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=MAX_EPOCHS,
    callbacks=[early_stopping],
    verbose=1,
)

print("Epochs actually completed:", len(history.history["loss"]))
print("Best monitored validation loss:", min(history.history["val_loss"]))

## 14. Evaluate training and internal validation performance

For an interpretable train-versus-validation comparison, training accuracy is measured using a **non-augmented** iterator over the *same training subset* used during fitting. Validation is evaluated using its existing rescale-only iterator. Training accuracy substantially higher than validation accuracy can suggest overfitting, although the gap is only one diagnostic. The final numbers are computed from the trained model, never hardcoded.

In [ ]:
train_eval_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=VALIDATION_SPLIT,
)
train_eval_ds = train_eval_datagen.flow_from_directory(
    directory=str(TRAIN_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=False,
    seed=SEED,
)
assert train_eval_ds.class_indices == train_ds.class_indices
assert set(train_eval_ds.filepaths) == set(train_ds.filepaths)

train_loss, train_accuracy = model.evaluate(train_eval_ds, verbose=1)
val_loss, val_accuracy = model.evaluate(val_ds, verbose=1)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Training Loss: {train_loss:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

accuracy_gap = train_accuracy - val_accuracy
print(f"Accuracy gap (training - validation): {accuracy_gap:+.4f}")
if accuracy_gap > 0.10:
    print("Interpretation: the gap exceeds 10 percentage points; this may indicate overfitting.")
elif accuracy_gap < -0.10:
    print("Interpretation: validation accuracy is much higher; review data differences and training augmentation.")
else:
    print("Interpretation: no large accuracy gap by the illustrative 10-point rule; inspect learning curves too.")

## 15. Plot accuracy and loss learning curves

Each graph plots values **actually recorded** in `history.history`, with epoch on the x-axis and both training and validation series plus a legend. A widening gap between training and validation performance can reveal overfitting; persistent low accuracy can signal underfitting.

In [ ]:
epochs_ran = range(1, len(history.history["accuracy"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history.history["accuracy"], marker="o", label="Training Accuracy")
axes[0].plot(epochs_ran, history.history["val_accuracy"], marker="o", label="Validation Accuracy")
axes[0].set_title("CNN Training vs Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history.history["loss"], marker="o", label="Training Loss")
axes[1].plot(epochs_ran, history.history["val_loss"], marker="o", label="Validation Loss")
axes[1].set_title("CNN Training vs Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 16. Optional extra: evaluate the original holdout validation and test directories

The assignment specifically requests training and internal-validation evaluation, completed above. As a stronger final check, use the *original* `validation/` and `test/` directories here **only after model training is finished**. Both use rescaling only, no augmentation, and the same class order. Do not tune the model on the test results.

In [ ]:
holdout_datagen = ImageDataGenerator(rescale=1.0 / 255.0)
class_names = list(train_ds.class_indices)

original_val_ds = holdout_datagen.flow_from_directory(
    directory=str(VALIDATION_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=class_names,
    shuffle=False,
    seed=SEED,
)
test_ds = holdout_datagen.flow_from_directory(
    directory=str(TEST_DIR),
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=class_names,
    shuffle=False,
    seed=SEED,
)
assert original_val_ds.class_indices == train_ds.class_indices
assert test_ds.class_indices == train_ds.class_indices

holdout_val_loss, holdout_val_accuracy = model.evaluate(original_val_ds, verbose=1)
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f"Original validation holdout — loss: {holdout_val_loss:.4f}, accuracy: {holdout_val_accuracy:.4f}")
print(f"Final test holdout — loss: {test_loss:.4f}, accuracy: {test_accuracy:.4f}")

## 17. Export reproducible results from the actual Kaggle run

This cell creates small CSV files in `/kaggle/working` after successful training. Include them in the GitHub repository only after they genuinely exist. Keep large image datasets and model binaries out of GitHub.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_history_df = pd.DataFrame(history.history)
training_history_df.index = training_history_df.index + 1
training_history_df.index.name = "epoch"
training_history_df.to_csv(OUTPUT_DIR / "training_history.csv")

metrics_df = pd.DataFrame([{
    "train_loss": float(train_loss),
    "train_accuracy": float(train_accuracy),
    "internal_val_loss": float(val_loss),
    "internal_val_accuracy": float(val_accuracy),
    "original_val_loss": float(holdout_val_loss),
    "original_val_accuracy": float(holdout_val_accuracy),
    "test_loss": float(test_loss),
    "test_accuracy": float(test_accuracy),
    "epochs_completed": len(history.history["loss"]),
}])
metrics_df.to_csv(OUTPUT_DIR / "evaluation_metrics.csv", index=False)
print("Saved actual run metrics to:", OUTPUT_DIR / "evaluation_metrics.csv")
print("Saved actual training history to:", OUTPUT_DIR / "training_history.csv")

## 18. Final assignment checklist

- ✅ Read dataset, count all three splits, merge tables and plot distribution.
- ✅ Apply train-only image augmentation and rescale-only validation.
- ✅ Build exactly three convolution/max-pooling blocks plus Flatten, Dense(512), Dropout(0.5), Dense(15)/Softmax.
- ✅ Compile with Adam, categorical crossentropy, accuracy.
- ✅ Train for at most 20 epochs with EarlyStopping monitoring validation loss, patience=5, restore best weights.
- ✅ Evaluate training and validation, discuss the accuracy gap and plot training history.
- ✅ Keep original validation and test splits separated until post-training holdout evaluation.

**Execution note:** This repository stores notebook code without fabricated outputs. Run all cells on Kaggle, save an executed notebook version and download small metrics CSVs if you want a repository containing confirmed experimental evidence.